# Apply Saved Transforms to New Images

For each slide:
1. Load seg image
2. Pad by `PADDING * scale` pixels (mirroring what alignment did at reg level)
3. Resize padded image to match padded anchor canvas (mirrors `_match_size`)
4. Apply scaled affine transform
5. Apply scaled elastic displacement field

In [ ]:
import sys, os
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path
from PIL import Image

ALIGNMENT_DIR = os.path.dirname(os.path.abspath("__file__"))
if ALIGNMENT_DIR not in sys.path:
    sys.path.insert(0, ALIGNMENT_DIR)

from alignment_pipeline import load_transform, discover_slides
from registration import read_ndpi_level, apply_affine, apply_elastic_transform, get_level_dimensions

## Configuration

In [ ]:
ALIGNMENT_OUTPUT_FOLDER = "path/alignment/was/saved/to"

NEW_IMAGE_FOLDER     = "/files/to/align/dir"
NEW_IMAGE_EXT        = ".png"
WARPED_OUTPUT_FOLDER = "/output"

ORIGINAL_SLIDE_FOLDER = "path/to/WSIs/dir/alignment/was/calculated/on"
ORIGINAL_SLIDE_EXT    = ".ndpi"
REGISTRATION_LEVEL    = 5 # ome tif level

# Padding used during alignment (pixels at registration level)
PADDING        = 200

# Seg images are level-0 / SEG_DOWNSAMPLE
SEG_DOWNSAMPLE = 16

FILL_VALUE   = 0
IS_LABEL_MAP = True

## Setup

In [ ]:
def read_image(path: str) -> np.ndarray:
    ext = Path(path).suffix.lower()
    if ext == ".ndpi":
        return read_ndpi_level(path, level=4)
    elif ext in (".tif", ".tiff"):
        return tifffile.imread(path)
    else:
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            raise IOError(f"Could not read {path}")
        if img.ndim == 3 and img.shape[2] == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        elif img.ndim == 3 and img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGBA)
        return img


def scale_affine_matrix(M: np.ndarray, sx: float, sy: float) -> np.ndarray:
    """Lift 2x3 affine M from reg frame to seg frame: M_seg = S @ M @ S_inv."""
    M = np.array(M, dtype=np.float64)
    if M.shape == (3, 3):
        M = M[:2, :]
    S     = np.array([[sx, 0, 0], [0, sy, 0], [0, 0, 1]], dtype=np.float64)
    S_inv = np.array([[1/sx, 0, 0], [0, 1/sy, 0], [0, 0, 1]], dtype=np.float64)
    return (S @ np.vstack([M, [0, 0, 1]]) @ S_inv)[:2, :]


def scale_displacement_field(disp: np.ndarray, sx: float, sy: float,
                              out_h: int, out_w: int) -> np.ndarray:
    """Resize displacement field and scale magnitudes."""
    dx = cv2.resize(disp[:, :, 0], (out_w, out_h), interpolation=cv2.INTER_LINEAR) * sx
    dy = cv2.resize(disp[:, :, 1], (out_w, out_h), interpolation=cv2.INTER_LINEAR) * sy
    return np.stack([dx, dy], axis=-1)

## Load transforms and compute scale

In [ ]:
affine_dir = Path(ALIGNMENT_OUTPUT_FOLDER) / "transforms" / "affine"
transforms_by_stem = {p.stem: load_transform(str(affine_dir), p.stem)
                      for p in sorted(affine_dir.glob("*.pkl"))}

original_slides = discover_slides(ORIGINAL_SLIDE_FOLDER, ORIGINAL_SLIDE_EXT)
slides_by_stem  = {Path(s).stem: s for s in original_slides}

# Anchor
anchor_stem = next(stem for stem, t in transforms_by_stem.items()
                   if t.get("notes", "").startswith("anchor"))
print(f"Anchor: {anchor_stem}")

# Anchor dims at reg level and level-0
anchor_dims = get_level_dimensions(slides_by_stem[anchor_stem])
W_reg, H_reg = anchor_dims[REGISTRATION_LEVEL]
W0,    H0    = anchor_dims[0]
print(f"Anchor reg-level dims (unpadded): W={W_reg}  H={H_reg}")
print(f"Anchor level-0 dims             : W={W0}  H={H0}")

# Scale from reg-level pixels to seg-level pixels (content only, no padding)
sx = (W0 / SEG_DOWNSAMPLE) / W_reg
sy = (H0 / SEG_DOWNSAMPLE) / H_reg
print(f"Scale reg→seg                   : sx={sx:.4f}  sy={sy:.4f}")

# Padding in seg-level pixels
pad_seg = int(round(PADDING * sx))   # sx ≈ sy for square pixels
print(f"Padding: {PADDING}px @ reg → {pad_seg}px @ seg")

# Expected canvas size = unpadded_seg + 2*pad_seg
CANVAS_W = int(round(W0 / SEG_DOWNSAMPLE)) + 2 * pad_seg
CANVAS_H = int(round(H0 / SEG_DOWNSAMPLE)) + 2 * pad_seg
print(f"Padded seg canvas               : W={CANVAS_W}  H={CANVAS_H}")

# Verify against the saved anchor aligned PNG
anchor_png = Path(ALIGNMENT_OUTPUT_FOLDER) / "aligned" / f"{anchor_stem}.png"
_chk = np.array(Image.open(str(anchor_png))).shape[:2]
print(f"Reg canvas from anchor PNG      : H={_chk[0]}  W={_chk[1]}")
CANVAS_W_REG, CANVAS_H_REG = _chk[1], _chk[0]
# True sx/sy mapping padded reg → padded seg (used to scale M and displacement)
sx_canvas = CANVAS_W / CANVAS_W_REG
sy_canvas = CANVAS_H / CANVAS_H_REG
print(f"Canvas scale (padded/padded)    : sx={sx_canvas:.6f}  sy={sy_canvas:.6f}")

## Apply transforms

In [ ]:
interp = cv2.INTER_NEAREST if IS_LABEL_MAP else cv2.INTER_LINEAR
out_dir = Path(WARPED_OUTPUT_FOLDER)
out_dir.mkdir(parents=True, exist_ok=True)

all_seg_slides = discover_slides(NEW_IMAGE_FOLDER, NEW_IMAGE_EXT)
seg_by_stem    = {Path(s).stem: s for s in all_seg_slides}
print(f"Found {len(all_seg_slides)} seg images\n")

for stem, t in sorted(transforms_by_stem.items(),
                       key=lambda x: x[1].get("slide_index", 0)):
    seg_path = seg_by_stem.get(stem)
    if seg_path is None:
        print(f"  [{stem}] no seg image found — skipping")
        continue

    notes = t.get("notes", "")
    if "skipped" in notes:
        print(f"  [{stem}] skipped slide — no output")
        continue

    print(f"  [{stem}] processing ...", end=" ", flush=True)

    # 1. Load
    img = read_image(seg_path)

    # 2. Pad with FILL_VALUE (mirrors _pad_image in alignment)
    img = cv2.copyMakeBorder(img, pad_seg, pad_seg, pad_seg, pad_seg,
                              cv2.BORDER_CONSTANT, value=FILL_VALUE)

    # 3. Resize padded image to match padded anchor canvas (mirrors _match_size)
    if img.shape[:2] != (CANVAS_H, CANVAS_W):
        img = cv2.resize(img, (CANVAS_W, CANVAS_H), interpolation=interp)

    # 4. Apply affine (scaled from reg canvas → seg canvas)
    M = t.get("affine_matrix")
    if M is not None:
        bv = (FILL_VALUE,) * img.shape[2] if img.ndim == 3 else FILL_VALUE
        img = apply_affine(img, scale_affine_matrix(M, sx_canvas, sy_canvas),
                           (CANVAS_H, CANVAS_W), border_value=bv, interpolation=interp)

    # 5. Apply elastic displacement field (scaled)
    disp = t.get("elastic_transform")
    if disp is not None:
        disp_s = scale_displacement_field(disp, sx_canvas, sy_canvas, CANVAS_H, CANVAS_W)
        img = apply_elastic_transform(img, disp_s, (CANVAS_H, CANVAS_W),
                                      default_value=float(FILL_VALUE), is_label=IS_LABEL_MAP)

    Image.fromarray(img).save(str(out_dir / f"{stem}.png"))
    print(f"done → {img.shape[:2]}")

print(f"\nAll done.")

## Visualize

In [ ]:
warped_files = sorted(out_dir.glob("*.png"))
n_show = min(len(warped_files), 8)
if n_show == 0:
    print("No warped files found.")
else:
    fig, axes = plt.subplots(1, n_show, figsize=(4 * n_show, 4))
    if n_show == 1: axes = [axes]
    for ax, f in zip(axes, warped_files[:n_show]):
        ax.imshow(np.array(Image.open(str(f))), interpolation="nearest")
        ax.set_title(f.stem, fontsize=9); ax.axis("off")
    plt.suptitle("Warped segmentation masks", fontsize=14)
    plt.tight_layout(); plt.show()

## Side-by-side: H&E aligned vs segmentation warped

In [ ]:
he_by_stem  = {f.stem: f for f in sorted((Path(ALIGNMENT_OUTPUT_FOLDER) / "aligned").glob("*.png"))}
seg_by_stem_out = {f.stem: f for f in sorted(out_dir.glob("*.png"))}
common_stems = [s for s in he_by_stem if s in seg_by_stem_out]

n_show = min(len(common_stems), 4)
if n_show == 0:
    print("No matching pairs found.")
else:
    fig, axes = plt.subplots(2, n_show, figsize=(4 * n_show, 8))
    if n_show == 1: axes = axes[:, np.newaxis]
    for k, stem in enumerate(common_stems[:n_show]):
        axes[0, k].imshow(np.array(Image.open(str(he_by_stem[stem]))))
        axes[0, k].set_title(f"H&E {stem}", fontsize=8); axes[0, k].axis("off")
        axes[1, k].imshow(np.array(Image.open(str(seg_by_stem_out[stem]))), interpolation="nearest")
        axes[1, k].set_title(f"Seg {stem}", fontsize=8); axes[1, k].axis("off")
    plt.suptitle("H&E aligned (top) vs Seg warped (bottom)", fontsize=12)
    plt.tight_layout(); plt.show()

    print(f"\n{'Stem':<50} {'H&E':>12} {'Seg':>12} {'Match?'}")
    print("-" * 85)
    for stem in common_stems:
        hs = np.array(Image.open(str(he_by_stem[stem]))).shape[:2]
        ss = np.array(Image.open(str(seg_by_stem_out[stem]))).shape[:2]
        print(f"{stem:<50} {str(hs):>12} {str(ss):>12}  {'✓' if hs == ss else '✗ MISMATCH'}")